# 11 - Hidden-like validation regimes (C1 / C2 / C3)

* **C1** reference available: truth in the pool, other reference spectra of the truth remain (self + T1/T2 excluded by `test_simulated_strict`).
* **C2** structure known, spectra hidden: every reference of the truth connectivity removed, truth stays in the pool. **Main regime.**
* **C3** structure absent: spectra removed AND truth removed from the pool (diagnostic only).

Regimes are assigned per truth connectivity by a seeded hash (persisted, never re-randomized); hiding is fold-scoped. Queries are TRAINING spectra (`dev_queries`); the visible `test.parquet` is never used here.

C2 eligibility needs the external universe (notebook 12): a C2 truth must also exist in a non-TRAIN source. Without it this notebook writes a **PRELIMINARY** table (C2 relaxed, shortcut risk flagged) -- re-run it with `REBUILD = True` after notebook 12. CPU only.

In [ ]:
import os, sys, json
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_ROOT = Path(os.environ.setdefault('ENVEDA_DRIVE_ROOT', '/content/drive/MyDrive/EnvedaCASMI'))
REPO_ROOT = Path(os.environ.setdefault('ENVEDA_REPO_ROOT', str(DRIVE_ROOT / 'repo')))
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from casmi.workspace.config import load_v2_config, input_path
from casmi.workspace.environment import set_seeds, system_report
CFG, P = load_v2_config(REPO_ROOT / 'configs' / 'casmi_v2_colab.yaml')
P.ensure()
SEED = set_seeds(CFG['runtime']['seed'])
NOTEBOOK = '11_colab_hidden_like_validation'
REBUILD = False   # True: rebuild + overwrite the persisted regime table (e.g. after notebook 12)
print(system_report())

In [ ]:
import numpy as np, pandas as pd
DEV_COLS = ['query_id', 'true_connectivity_key', 'fold', 'neutral_mass', 'ionization_mode', 'adduct', 'instrument_type', 'exact_mass_true']
dev = pd.read_parquet(input_path(CFG, P, 'dev_queries'), columns=DEV_COLS)
dev['query_id'] = dev['query_id'].astype(str)
tsm = pd.read_parquet(input_path(CFG, P, 'train_spectrum_metadata'), columns=['train_spectrum_id', 'connectivity_key'])
REF_CONN = tsm['connectivity_key'].astype(str).to_numpy()
folds = pd.read_parquet(input_path(CFG, P, 'connectivity_folds'))
print(f'dev queries: {len(dev):,} | reference spectra: {len(REF_CONN):,} | fold rows: {len(folds):,}')

# the visible competition test is NEVER a validation input: every query must be a TRAINING spectrum
not_train = ~dev['query_id'].isin(set(tsm['train_spectrum_id'].astype(str)))
assert not not_train.any(), f'{int(not_train.sum())} dev queries are not training spectra'
del tsm

In [ ]:
UNIVERSE_ROOT = P.candidate_db_dir / 'universe'
HAS_UNIVERSE = (UNIVERSE_ROOT / 'universe_manifest.json').exists()
if HAS_UNIVERSE:
    from casmi.candidates.universe import load_candidate_keys, keys_to_ids, read_candidates
    UNIVERSE_MANIFEST = json.loads((UNIVERSE_ROOT / 'universe_manifest.json').read_text())
    UKEYS = load_candidate_keys(UNIVERSE_ROOT)   # sorted S14, memory-mapped
    tid = keys_to_ids(UKEYS, dev['true_connectivity_key'].unique())
    UNIFIED_TRUTHS = read_candidates(UNIVERSE_ROOT, tid[tid >= 0], columns=['connectivity_key', 'candidate_sources'])
    C2_MODE = 'external_universe'
else:
    UNIVERSE_MANIFEST = None
    mv = pd.read_parquet(input_path(CFG, P, 'molecule_mass_variants'), columns=['connectivity_key'])
    UKEYS = np.sort(mv['connectivity_key'].astype(str).unique())
    UNIFIED_TRUTHS = None
    C2_MODE = 'PRELIMINARY_train_only_universe'
print('universe available:', HAS_UNIVERSE, '| C2 mode:', C2_MODE, '| universe size:', f'{len(UKEYS):,}')

In [ ]:
from casmi.validation.regimes import RegimeConfig, build_regime_table, regime_counts, truth_eligibility
from casmi.workspace.artifacts import write_metadata, read_metadata

v = CFG['validation']
RC = RegimeConfig(split_seed=v['split_seed'], regime_fractions=v['regime_fractions'], hidden_scope=v['hidden_scope'],
                  c1_reference_protocol=v['c1_reference_protocol'], c1_min_truth_spectra=v['c1_min_truth_spectra'],
                  c2_require_external_source=bool(v['c2_require_external_source']) and HAS_UNIVERSE,
                  max_queries_per_regime_per_fold=v['max_queries_per_regime_per_fold'])
SIGNATURE = {'regime_config_hash': RC.config_hash(), 'c2_mode': C2_MODE,
             'universe_n_candidates': (UNIVERSE_MANIFEST or {}).get('n_candidates'), 'n_dev_queries': int(len(dev))}

REG_PATH = P.processed_dir / 'validation_regimes.parquet'
META_NAME = 'validation_regimes.metadata.json'
old = read_metadata(P.processed_dir, META_NAME)
if REG_PATH.exists() and old and old.get('signature') == SIGNATURE and not REBUILD:
    REG = pd.read_parquet(REG_PATH)
    print('loaded persisted regime table (same signature)')
elif REG_PATH.exists() and not REBUILD:
    raise RuntimeError(f'persisted regimes have a different signature {old and old.get("signature")} != {SIGNATURE}; set REBUILD = True to overwrite')
else:
    REG = build_regime_table(dev, REF_CONN, unified=UNIFIED_TRUTHS, cfg=RC)
    print('built regime table')
print(REG['regime'].fillna('UNASSIGNED').value_counts())

In [ ]:
COUNTS = regime_counts(REG)
display(COUNTS.pivot(index='fold', columns='regime', values='n_queries').fillna(0).astype(int))
ELIG = truth_eligibility(REG['true_connectivity_key'].unique(), REF_CONN, UNIFIED_TRUTHS, RC)
ELIG_SUMMARY = {'n_truth_connectivities': int(len(ELIG)),
                'eligible_C1_rate': float(ELIG['eligible_C1'].mean()),
                'eligible_C2_rate': float(ELIG['eligible_C2'].mean()),
                'c2_ineligible_reasons': ELIG.loc[~ELIG['eligible_C2'], 'c2_ineligible_reason'].value_counts().to_dict()}
print(json.dumps(ELIG_SUMMARY, indent=2))

In [ ]:
# leakage + masking invariants, every fold (raises on any failure)
from casmi.validation.regimes import assert_regime_integrity
CHECKS = pd.concat([assert_regime_integrity(REG, folds, REF_CONN, UKEYS, f, RC) for f in sorted(REG['fold'].unique())], ignore_index=True)
display(CHECKS)
print('ALL REGIME CHECKS PASSED:', bool(CHECKS['passed'].all()))

In [ ]:
import matplotlib.pyplot as plt
d = REG.merge(dev[['query_id', 'neutral_mass', 'ionization_mode']], on='query_id', how='left')
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
COUNTS.pivot(index='fold', columns='regime', values='n_queries').fillna(0).plot.bar(ax=axes[0], title='queries per fold / regime')
for r, g in d.groupby(d['regime'].fillna('UNASSIGNED')):
    axes[1].hist(np.log10(g['n_truth_spectra'].clip(lower=1)), bins=40, histtype='step', label=r)
    axes[2].hist(g['neutral_mass'].dropna(), bins=60, histtype='step', label=r)
axes[1].set_xlabel('log10 #spectra of the truth connectivity'); axes[1].legend()
axes[2].set_xlabel('query neutral mass (Da)'); axes[2].legend()
fig.tight_layout()
FIG_DIR = P.reports_dir / 'validation'
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIG_DIR / 'regime_distributions.png', dpi=120)
display(pd.crosstab(d['regime'].fillna('UNASSIGNED'), d['ionization_mode'].astype(str), normalize='index').round(3))

In [ ]:
from casmi.validation.reporting import save_report
REG.to_parquet(REG_PATH, index=False)
write_metadata(P.processed_dir, NOTEBOOK, config={'validation': CFG['validation'], 'regime_config': RC.__dict__}, seed=RC.split_seed,
               input_paths={k: input_path(CFG, P, k) for k in ('dev_queries', 'train_spectrum_metadata', 'connectivity_folds')},
               schema_version=RC.version, repo_root=REPO_ROOT, filename=META_NAME,
               extra={'signature': SIGNATURE, 'c2_mode': C2_MODE, 'n_rows': int(len(REG))})
summary_table = COUNTS.assign(c2_mode=C2_MODE, regime_config_hash=RC.config_hash())
save_report(summary_table, {'notebook': NOTEBOOK, 'c2_mode': C2_MODE, 'signature': SIGNATURE, 'eligibility': ELIG_SUMMARY,
                            'counts': COUNTS.to_dict(orient='records'), 'checks_passed': bool(CHECKS['passed'].all()),
                            'note': 'regime definitions + counts only; no model metric is computed in this notebook'},
            P.reports_dir, 'validation_summary')
CHECKS.to_parquet(P.reports_dir / 'validation' / 'regime_checks.parquet', index=False)
print('saved', REG_PATH, 'and', P.reports_dir / 'validation_summary.parquet')